# Parquet - Duplicates

Introduce exact and/or conflicting-key duplicates without treating shared timestamps as business keys.

**Instructions:** Run `prepare_data_parquet.ipynb` to create clean `data/weather.parquet` first (`uv add pyarrow`). This notebook overwrites the Parquet file; restore the clean artifact before rerunning. For combination, use only the required config/function/call, and load/save once.

In [ ]:
from pathlib import Path
from random import Random

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

# Configuration
DATA_PATH = Path("data/weather.parquet")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local dates.
SEED = 814205

# Duplicates
KEY_COLUMNS = ["time"]
DUPLICATE_KIND = "mixed"  # exact, conflicting, mixed, random
DUPLICATE_PLACEMENT = "mixed"  # scattered, grouped, mixed, random
DUPLICATE_RATE = (0.01, 0.04)  # Share of eligible original rows copied once.
DUPLICATE_GROUPS = 3
DUPLICATE_GROUPED_SHARE = 0.70
DUPLICATE_CONFLICT_COLUMN = "temperature_2m"
DUPLICATE_CONFLICT_OFFSET = 1.5
DUPLICATE_CONFLICT_SHARE = 0.50

In [ ]:
# Shared selection: include once in an assembled dirty_data.py.
def scoped_rows(df, *, time_column, timezone, start_date, end_date):
    if time_column not in df:
        raise ValueError("Time column not found.")
    times = pd.to_datetime(df[time_column], utc=True, errors="raise")
    if times.isna().any():
        raise ValueError("Time column contains missing timestamps.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")
    scoped = df.loc[times.ge(start) & times.lt(end)]
    if scoped.empty:
        raise ValueError("No records in the selected interval.")
    # Indices refer to the input DataFrame. No complete-hourly assumption.
    return scoped.sort_values(time_column, kind="stable").index.tolist(), start, end

def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_duplicates(
    df, rng, *, time_column, timezone, start_date, end_date,
    key_columns, duplicate_kind, placement, duplicate_rate,
    n_groups, grouped_share, conflict_column, conflict_offset, conflict_share,
):
    if not key_columns or any(c not in df for c in key_columns):
        raise ValueError("Expected existing key columns.")
    clean = df.reset_index(drop=True).copy()
    if clean[key_columns].isna().any().any() or clean.duplicated(subset=key_columns).any():
        raise ValueError("Expected unique, nonmissing record keys.")
    eligible, _, _ = scoped_rows(
        clean, time_column=time_column, timezone=timezone,
        start_date=start_date, end_date=end_date,
    )
    if duplicate_kind not in ("exact", "conflicting", "mixed", "random"):
        raise ValueError("Unknown duplicate kind.")
    if not 0 < conflict_share < 1:
        raise ValueError("Invalid conflict share.")
    kind = (rng.choice(("exact", "conflicting", "mixed"))
            if duplicate_kind == "random" else duplicate_kind)
    if kind != "exact":
        if (conflict_column not in clean or conflict_column in key_columns
                or conflict_column == time_column or conflict_offset == 0):
            raise ValueError("Provide a non-key numeric conflict column and nonzero offset.")
        if not pd.api.types.is_numeric_dtype(clean[conflict_column]):
            raise ValueError("Conflicting copies require a numeric column.")
        if pd.api.types.is_integer_dtype(clean[conflict_column]) and not float(conflict_offset).is_integer():
            raise ValueError("Use an integer conflict offset to preserve Parquet integer type.")
        eligible = [i for i in eligible if pd.notna(clean.at[i, conflict_column])]
    selected, _ = select_positions(
        len(eligible), rng, method=placement, rate=duplicate_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    copies = clean.loc[[eligible[i] for i in selected]].copy().reset_index(drop=True)
    if kind == "mixed" and len(copies) < 2:
        raise ValueError("Mixed duplicates need at least two copies.")
    conflicts = (0 if kind == "exact" else len(copies) if kind == "conflicting" else
                 max(1, min(len(copies) - 1, round(len(copies) * conflict_share))))
    if conflicts:
        for i in rng.sample(range(len(copies)), k=conflicts):
            copies.at[i, conflict_column] = copies.at[i, conflict_column] + conflict_offset
    # Sort original records before their copies, preserving timestamp order.
    result = pd.concat([clean, copies], ignore_index=True)
    sort_columns = [time_column, *[c for c in key_columns if c != time_column]]
    result = result.sort_values(sort_columns, kind="stable").reset_index(drop=True)
    if result.duplicated(subset=key_columns).sum() != len(copies):
        raise AssertionError("Unexpected repeated-key count.")
    return result

In [ ]:
# Small controlled trial (no source file changed).
trial_times = pd.date_range("2025-08-01", periods=7*24, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = pd.DataFrame({
    "time": trial_times,
    "temperature_2m": [float(10 + i % 17) for i in range(len(trial_times))],
    "relative_humidity_2m": pd.Series([int(40 + i % 40) for i in range(len(trial_times))], dtype="int64"),
    "precipitation": [float(i % 10) for i in range(len(trial_times))],
})

trial_dirty = add_duplicates(trial, Random(SEED),
    time_column="time", timezone=TIMEZONE, start_date="2025-08-01", end_date="2025-08-08",
    key_columns=["time"], duplicate_kind="mixed", placement="mixed",
    duplicate_rate=(0.10, 0.10), n_groups=2, grouped_share=0.70,
    conflict_column="temperature_2m", conflict_offset=1.5, conflict_share=0.50)
assert len(trial_dirty) - len(trial) == round(len(trial) * 0.10)
assert trial_dirty.duplicated(subset=["time"]).sum() == round(len(trial) * 0.10)
trial_dirty.loc[trial_dirty.duplicated(subset=["time"], keep=False)].head(10)

In [ ]:
# Verify actual Parquet types and values on a small temporary file.
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    clean_path = Path(temporary) / "clean.parquet"
    temp_path = Path(temporary) / "trial.parquet"
    trial.to_parquet(clean_path, index=False)
    trial_dirty.to_parquet(temp_path, index=False)
    roundtrip = pd.read_parquet(temp_path)
    before_schema = pq.read_schema(clean_path)
    after_schema = pq.read_schema(temp_path)
    assert len(roundtrip) == len(trial_dirty)
    assert roundtrip.columns.tolist() == trial_dirty.columns.tolist()
    assert pq.read_metadata(temp_path).num_rows == len(trial_dirty)
    assert all(before_schema.field(c).type == after_schema.field(c).type for c in trial.columns)
    print("Stored Parquet schema:", after_schema)

In [ ]:
# Apply the reusable function to a freshly prepared Parquet artifact.
clean = pd.read_parquet(DATA_PATH)
rng = Random(SEED)
dirty = add_duplicates(
    clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    key_columns=KEY_COLUMNS, duplicate_kind=DUPLICATE_KIND,
    placement=DUPLICATE_PLACEMENT, duplicate_rate=DUPLICATE_RATE,
    n_groups=DUPLICATE_GROUPS, grouped_share=DUPLICATE_GROUPED_SHARE,
    conflict_column=DUPLICATE_CONFLICT_COLUMN, conflict_offset=DUPLICATE_CONFLICT_OFFSET,
    conflict_share=DUPLICATE_CONFLICT_SHARE,
)
dirty.to_parquet(DATA_PATH, index=False)
saved = pd.read_parquet(DATA_PATH)

In [ ]:
# Inspect: whole-row duplication versus repeated intended keys.
saved.duplicated().sum(), saved.duplicated(subset=KEY_COLUMNS).sum()

In [ ]:
# Verify persistence: all changes exist after writing the Parquet file.
assert len(saved) == len(dirty)
assert saved.columns.tolist() == dirty.columns.tolist()
assert pq.read_metadata(DATA_PATH).num_rows == len(saved)
assert len(saved) > len(clean)
assert saved.duplicated(subset=KEY_COLUMNS).sum() == len(saved) - len(clean)
print("Extra copied records:", len(saved) - len(clean))

In [ ]:
# Added duplicate-key records by local date.
duplicated = saved.loc[saved.duplicated(subset=KEY_COLUMNS, keep="first")].copy()
dates = pd.to_datetime(duplicated[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
counts = dates.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3))
counts.plot.bar(ax=ax)
ax.set(title="Extra duplicate-key records by date", xlabel="Local date", ylabel="Copies")
plt.tight_layout(); plt.show()